# BLAST ho.report analysis

Parse `reports/ho.report` and plot **finalObj score** vs **iteration** (one point per `input` line).

Set `RUN_SUBDIR` below to a run folder under AgenticBLAST (e.g. `ML-Tersoff-1_PE`).

Outputs (written under the run folder):
- `reports/score_vs_iteration.png`
- `reports/score_vs_iteration.csv`

In [ ]:
import csv
import re
from pathlib import Path

import matplotlib.pyplot as plt

# Run folder relative to this notebook (AgenticBLAST root or a subfolder)
RUN_SUBDIR = "ML-Tersoff-1_PE"  # or "ML-Tersoff-1_PE_12"

ROOT = Path(".").resolve()
WORKDIR = (ROOT / RUN_SUBDIR).resolve() if RUN_SUBDIR else ROOT
REPORT_PATH = WORKDIR / "reports" / "ho.report"
PLOT_PATH = WORKDIR / "reports" / "score_vs_iteration.png"
CSV_PATH = PLOT_PATH.with_suffix(".csv")

FINALOBJ_RE = re.compile(r"^\#\s*([\d.]+)\s*\|\s*finalObj\s*\|")
INPUT_RE = re.compile(r"^input\s+")

print(f"Root:    {ROOT}")
print(f"Workdir: {WORKDIR}")
print(f"Report:  {REPORT_PATH}")

In [ ]:
def parse_ho_report(report_path: Path) -> list[dict]:
    trials: list[dict] = []
    current: dict | None = None

    with report_path.open() as fh:
        for raw_line in fh:
            line = raw_line.rstrip("\n")

            if INPUT_RE.match(line):
                current = {
                    "iteration": len(trials) + 1,
                    "score": None,
                    "status": "pending",
                    "reason": "",
                }
                trials.append(current)
                continue

            if current is None:
                continue

            if "invalid parameter" in line:
                current["score"] = 1_000_000.0
                current["status"] = "invalid"
                current["reason"] = line.lstrip("# ").strip()
                current = None
                continue

            mobj = FINALOBJ_RE.match(line)
            if mobj:
                current["score"] = float(mobj.group(1))
                current["status"] = "final"
                tail = line.split("| finalObj |", 1)[-1].strip()
                current["reason"] = tail.split("DUMP", 1)[-1].strip() if "DUMP" in tail else tail
                current = None

    return trials

In [ ]:
if not REPORT_PATH.is_file():
    raise FileNotFoundError(f"Report not found: {REPORT_PATH}")

trials = parse_ho_report(REPORT_PATH)
scored = [t for t in trials if t["score"] is not None]
scores = [t["score"] for t in scored]

print(f"Trials (input lines): {len(trials)}")
print(f"Scored trials:        {len(scored)}")
if scores:
    print(f"Best score:           {min(scores):.6g}")
    print(f"Worst score:          {max(scores):.6g}")

CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
with CSV_PATH.open("w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=["iteration", "score", "status", "reason"])
    writer.writeheader()
    writer.writerows(trials)
print(f"Wrote CSV: {CSV_PATH}")

In [ ]:
xs = [t["iteration"] for t in scored]
ys = scores

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(xs, ys, linewidth=0.8, alpha=0.85, color="#2563eb")
ax.set_xlabel("Iteration")
ax.set_ylabel("Score (finalObj)")
ax.set_title(f"BLAST objective score — {WORKDIR.name}")
ax.grid(True, alpha=0.3)

best = min(scores)
ax.axhline(best, color="#16a34a", linestyle="--", linewidth=1, label=f"Best = {best:.4g}")
ax.legend(loc="upper right")

fig.tight_layout()
fig.savefig(PLOT_PATH, dpi=150)
plt.show()
print(f"Wrote plot: {PLOT_PATH}")